In [4]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import train_test_split
import scipy.special

def generate_features(n_samples=1000):
    """
    Generate mixed data types: 15 continuous, 5 discrete, 5 categorical
    """
    # generate continuous features
    rho = 0.8   
    mus = np.random.uniform(10, 50, 15)
    sigmas = np.random.uniform(2, 10, 15)

    # covariance matrix
    cov = np.zeros((15, 15))
    np.fill_diagonal(cov, sigmas**2)
    for i in range(15):
        for j in range(i+1, 15):
            cov[i, j] = cov[j, i] = rho * sigmas[i] * sigmas[j]
            
    continuous_data = np.random.multivariate_normal(mus, cov, size=n_samples)
    continuous_features = {}
    for i in range(15):
        continuous_features[f'continuous_{i}'] = continuous_data[:, i]

    # generate discrete features
    discrete_features = {}
    lambdas = [2, 3, 4, 5, 6] 
    for i in range(5):
        discrete_features[f'discrete_{i}'] = np.random.poisson(lambdas[i], n_samples)

    # generate categorical features
    categorical_features = {}
    for i in range(5):
        if i < 4:
            # 4 binary features 
            categories = ['No', 'Yes']
            prob = np.random.uniform(0.3, 0.7)
            probs = [1-prob, prob]
            categorical_features[f'categorical_{i}'] = np.random.choice(categories, size=n_samples, p=probs)
        else: 
            # 1 multinomial feature 
            categories = ['Level_A', 'Level_B', 'Level_C']
            probs = [0.4, 0.35, 0.25]  
            categorical_features[f'categorical_{i}'] = np.random.choice(categories, size=n_samples, p=probs)
    
    return continuous_features, discrete_features, categorical_features

def standardise_features(cont_features, disc_features, cat_features):
    """
    Standardise numerical features and one-hot encode categorical features.
    """
    # create initial dataframe
    df = pd.DataFrame(cont_features)
    for col, values in disc_features.items():
        df[col] = values
    for col, values in cat_features.items():
        df[col] = values
    
    # get column lists
    cont_cols = list(cont_features.keys())
    disc_cols = list(disc_features.keys())
    cat_cols = list(cat_features.keys())
    
    # standardise continuous and discrete features
    scaler = StandardScaler()
    df_scaled_cont_disc = pd.DataFrame(
        scaler.fit_transform(df[cont_cols + disc_cols]),
        columns=cont_cols + disc_cols
    )
    
    # one-hot encode categorical features
    encoder = OneHotEncoder(sparse_output=False, drop='first')  
    df_encoded_cat = pd.DataFrame(
        encoder.fit_transform(df[cat_cols]),
        columns=encoder.get_feature_names_out(cat_cols)
    )
    df_encoded_cat = df_encoded_cat.astype(int)
    
    # combine all features
    df_processed = pd.concat([df_scaled_cont_disc, df_encoded_cat], axis=1)
    
    return df_processed, cont_cols, disc_cols, cat_cols

def assign_feature_importance(df_processed):
    """
    Assign importance weights to features.
    """
    all_features = df_processed.columns.to_list()
    
    # define importance categories
    high_importance = ['continuous_0', 'continuous_3', 'continuous_12', 'discrete_1', 'discrete_3', 'categorical_0_Yes', 'categorical_2_Yes']
    medium_importance = ['continuous_1', 'continuous_5', 'continuous_9', 'continuous_11', 'continuous_13', 'discrete_0', 'discrete_4', 'categorical_1_Yes', 'categorical_4_Level_C']
    low_importance = [col for col in all_features if col not in high_importance + medium_importance]
    
    # assign weights with random signs
    def random_weight(low, high):
        weight = np.random.uniform(low, high)
        sign = np.random.choice([-1, 1])
        return np.round(sign * weight, 2)

    feature_weights = {}
    for f in high_importance:
        feature_weights[f] = random_weight(2.0, 3.0)
    for f in medium_importance:
        feature_weights[f] = random_weight(1.0, 2.0)
    for f in low_importance:
        feature_weights[f] = random_weight(0.0, 1.0)

    return feature_weights, high_importance, medium_importance, low_importance

def generate_target(df_processed, feature_weights):
    """
    Generate binary target variable using a logistic function.
    """
    all_features = df_processed.columns.to_list()
    X = df_processed[all_features].values
    B = np.array([feature_weights[f] for f in all_features])
    
    # mean predicted probability for a given intercept
    def calculate_mean_prob(intercept):
        logit_p = X @ B + intercept
        p = scipy.special.expit(logit_p)
        return p.mean()

    # binary search to balance classes
    low, high = -10.0, 10.0
    for _ in range(25):
        mid = (low + high)/2
        if calculate_mean_prob(mid) > 0.5:
            high = mid
        else:
            low = mid 
    intercept = (low + high) / 2

    # generate target
    logit_p = X @ B + intercept
    p = scipy.special.expit(logit_p)
    y = np.random.binomial(1, p)
    
    # add target to dataframe
    synthetic_data = df_processed.copy()
    synthetic_data['target'] = y
    
    return synthetic_data

def check_data_quality(synthetic_data):
    """
    Check data quality: types, target distribution balance, multicollinearity.
    """
    # check data types
    data_types = synthetic_data.dtypes
    
    # check target distribution
    target_balance = synthetic_data['target'].mean()
    
    # check multicollinearity
    X_vif = synthetic_data.drop(columns=['target'])
    vif_data = pd.DataFrame()
    vif_data['feature'] = X_vif.columns
    vif_data['VIF'] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
    max_vif = vif_data['VIF'].max()
    
    quality_report = {
        'target_balance': target_balance,
        'max_vif': max_vif,
        'num_features': len(synthetic_data.columns) - 1
    }
    
    return quality_report
    
def split_and_save_data(synthetic_data, iteration, output_dir="data/complete", random_state=57):
    """
    Split data into train and test sets and save to files.
    """
    # stratified split data
    train_data, test_data = train_test_split(
        synthetic_data, 
        test_size=0.3, 
        random_state=random_state, 
        stratify=synthetic_data['target']
    )
    
    # create iteration directory path
    iter_dir = f"{output_dir}/iteration_{iteration}"
    os.makedirs(iter_dir, exist_ok=True)
    
    # save files
    train_data.to_csv(f"{iter_dir}/train_complete.csv", index=False)
    test_data.to_csv(f"{iter_dir}/test_complete.csv", index=False)
    
    print(f"Train shape: {train_data.shape}, Test shape: {test_data.shape}")
    print(f"Train target distribution: {train_data['target'].value_counts(normalize=True)}")
    print(f"Test target distribution: {test_data['target'].value_counts(normalize=True)}")
    
    return train_data, test_data

def generate_synthetic_dataset(iteration, random_seed, n_samples=1000):
    """
    Generate a complete synthetic dataset.
    """
    print(f"Generating synthetic dataset for iteration {iteration} with seed {random_seed}")

    # set random seed for replicability
    np.random.seed(random_seed)
    
    # 1. generate features
    cont_features, disc_features, cat_features = generate_features(n_samples)
    # 2. scale and encode features
    df_processed, cont_cols, disc_cols, cat_cols = standardise_features(cont_features, disc_features, cat_features)
    # 3. assign importance weights
    feature_weights, high, medium, low = assign_feature_importance(df_processed)
    # 4. create the target
    synthetic_data = generate_target(df_processed, feature_weights)
    # 5. check data quality
    quality_report = check_data_quality(synthetic_data)
    print(f"Data quality report: {quality_report}")
    # 6. split and save
    train_data, test_data = split_and_save_data(
        synthetic_data, 
        iteration=iteration, 
        output_dir="data/complete", 
        random_state=random_seed
    )
    
    return train_data, test_data, synthetic_data

if __name__ == "__main__":
    for seed in range(1, 101):
    
        # generate dataset
        train_data, test_data, synthetic_data = generate_synthetic_dataset(
            iteration=seed, 
            random_seed=seed,
            n_samples=1000
        )
        print(f"\n")

    print(f"Successfully generated synthetic data for 100 iterations.")

Generating synthetic dataset for iteration 1 with seed 1
Data quality report: {'target_balance': 0.497, 'max_vif': 4.798924853721706, 'num_features': 26}
Train shape: (700, 27), Test shape: (300, 27)
Train target distribution: target
0    0.502857
1    0.497143
Name: proportion, dtype: float64
Test target distribution: target
0    0.503333
1    0.496667
Name: proportion, dtype: float64


Generating synthetic dataset for iteration 2 with seed 2
Data quality report: {'target_balance': 0.5, 'max_vif': 4.999901006269513, 'num_features': 26}
Train shape: (700, 27), Test shape: (300, 27)
Train target distribution: target
0    0.5
1    0.5
Name: proportion, dtype: float64
Test target distribution: target
0    0.5
1    0.5
Name: proportion, dtype: float64


Generating synthetic dataset for iteration 3 with seed 3
Data quality report: {'target_balance': 0.508, 'max_vif': 5.246798988180147, 'num_features': 26}
Train shape: (700, 27), Test shape: (300, 27)
Train target distribution: target
1    0

In [ ]:
# Linear predictors range 

import numpy as np
import scipy.special

def linear_predictor(df_processed, feature_weights):
    all_features = df_processed.columns.to_list()
    X = df_processed[all_features].values
    B = np.array([feature_weights[f] for f in all_features])
    
    def calculate_mean_prob(intercept):
        logit_p = X @ B + intercept
        p = scipy.special.expit(logit_p)
        return p.mean()

    # find intercept to balance classes
    low, high = -10.0, 10.0
    for _ in range(25):
        mid = (low + high) / 2
        if calculate_mean_prob(mid) > 0.5:
            high = mid
        else:
            low = mid
    intercept = (low + high) / 2

    eta = X @ B + intercept
    return eta

n_samples = 1000

for iteration in range(1, 101):
    np.random.seed(iteration)

    cont, disc, cat = generate_features(n_samples)
    df_processed, *_ = standardise_features(cont, disc, cat)
    feature_weights, *_ = assign_feature_importance(df_processed)

    eta = linear_predictor(df_processed, feature_weights)

    # basic range check
    count = ((eta >= -5) & (eta <= 5)).sum()

    print(f"\nSimulation {iteration}")
    print(f"Linear predictors in [-5, 5]: {count} / {len(eta)}")

In [ ]:
# Histogram of variables 

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

train_iter1 = pd.read_csv("data/complete/iteration_1/train_complete.csv")
test_iter1 = pd.read_csv("data/complete/iteration_1/test_complete.csv")

synthetic_data_iter1 = pd.concat([train_iter1, test_iter1], axis=0)

ax = synthetic_data_iter1.hist(figsize=(20,15), bins=30, color='steelblue', edgecolor='white', grid=True)

plt.suptitle("Histograms of Synthetic Data Variables (Simulation 1)", fontsize=16)

for axes_row in ax:
    for a in axes_row:
        a.set_ylabel("Frequency", fontsize=10)
        a.set_xlabel("Value", fontsize=10)
        a.set_axisbelow(True)

plt.tight_layout(rect=[0, 0, 1, 0.96])

plt.savefig("histogram_simulation1.png", dpi=300) 
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

train_iter1 = pd.read_csv("data/complete/iteration_1/train_complete.csv")
test_iter1 = pd.read_csv("data/complete/iteration_1/test_complete.csv")

synthetic_data_iter1 = pd.concat([train_iter1, test_iter1], axis=0)

cols = ["continuous_0", "discrete_0", "categorical_0_Yes", "target"]
df_plot = synthetic_data_iter1[cols]

ax = df_plot.hist(figsize=(20,4), bins=30, color='steelblue', edgecolor='white', grid=True, layout=(1,4))

for axes_row in ax:
    for a in axes_row:
        a.set_ylabel("Frequency", fontsize=10)
        a.set_xlabel("Value", fontsize=10)
        a.set_axisbelow(True)

plt.tight_layout(rect=[0, 0, 1, 0.96])

plt.savefig("histograms_simulation1_selected.png", dpi=300)
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

target_proportions = []

for i in range(1, 101):
    train = pd.read_csv(f"data/complete/iteration_{i}/train_complete.csv")
    test = pd.read_csv(f"data/complete/iteration_{i}/test_complete.csv")
    
    full = pd.concat([train, test])
    proportion_1 = full["target"].mean()
    target_proportions.append(prop_1)

plt.figure(figsize=(7,5))

plt.hist(target_proportions, bins=15, color="steelblue", edgecolor="white")

plt.title("Distribution of Target Across 100 Simulations", fontsize=14)
plt.xlabel("Proportion of observations with target = 1", fontsize=11)
plt.ylabel("Frequency", fontsize=11)

plt.grid(True)
plt.gca().set_axisbelow(True)   

plt.tight_layout()
plt.savefig("target_distribution.png", dpi=300)
plt.show()


In [ ]:
# Different correlation settings

In [ ]:
# rho = 0
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import train_test_split
import scipy.special

def generate_features(n_samples=1000):
    """
    Generate mixed data types: 15 continuous, 5 discrete, 5 categorical
    """
    # generate continuous features
    rho = 0
    mus = np.random.uniform(10, 50, 15)
    sigmas = np.random.uniform(2, 10, 15)

    # covariance matrix
    cov = np.zeros((15, 15))
    np.fill_diagonal(cov, sigmas**2)
    for i in range(15):
        for j in range(i+1, 15):
            cov[i, j] = cov[j, i] = rho * sigmas[i] * sigmas[j]
            
    continuous_data = np.random.multivariate_normal(mus, cov, size=n_samples)
    continuous_features = {}
    for i in range(15):
        continuous_features[f'continuous_{i}'] = continuous_data[:, i]

    # generate discrete features
    discrete_features = {}
    lambdas = [2, 3, 4, 5, 6] 
    for i in range(5):
        discrete_features[f'discrete_{i}'] = np.random.poisson(lambdas[i], n_samples)

    # generate categorical features
    categorical_features = {}
    for i in range(5):
        if i < 4:
            # 4 binary features 
            categories = ['No', 'Yes']
            prob = np.random.uniform(0.3, 0.7)
            probs = [1-prob, prob]
            categorical_features[f'categorical_{i}'] = np.random.choice(categories, size=n_samples, p=probs)
        else: 
            # 1 multinomial feature 
            categories = ['Level_A', 'Level_B', 'Level_C']
            probs = [0.4, 0.35, 0.25]  
            categorical_features[f'categorical_{i}'] = np.random.choice(categories, size=n_samples, p=probs)
    
    return continuous_features, discrete_features, categorical_features

def standardise_features(cont_features, disc_features, cat_features):
    """
    Standardise numerical features and one-hot encode categorical features.
    """
    # create initial dataframe
    df = pd.DataFrame(cont_features)
    for col, values in disc_features.items():
        df[col] = values
    for col, values in cat_features.items():
        df[col] = values
    
    # get column lists
    cont_cols = list(cont_features.keys())
    disc_cols = list(disc_features.keys())
    cat_cols = list(cat_features.keys())
    
    # standardise continuous and discrete features
    scaler = StandardScaler()
    df_scaled_cont_disc = pd.DataFrame(
        scaler.fit_transform(df[cont_cols + disc_cols]),
        columns=cont_cols + disc_cols
    )
    
    # one-hot encode categorical features
    encoder = OneHotEncoder(sparse_output=False, drop='first')  
    df_encoded_cat = pd.DataFrame(
        encoder.fit_transform(df[cat_cols]),
        columns=encoder.get_feature_names_out(cat_cols)
    )
    df_encoded_cat = df_encoded_cat.astype(int)
    
    # combine all features
    df_processed = pd.concat([df_scaled_cont_disc, df_encoded_cat], axis=1)
    
    return df_processed, cont_cols, disc_cols, cat_cols

def assign_feature_importance(df_processed):
    """
    Assign importance weights to features.
    """
    all_features = df_processed.columns.to_list()
    
    # define importance categories
    high_importance = ['continuous_0', 'continuous_3', 'continuous_12', 'discrete_1', 'discrete_3', 'categorical_0_Yes', 'categorical_2_Yes']
    medium_importance = ['continuous_1', 'continuous_5', 'continuous_9', 'continuous_11', 'continuous_13', 'discrete_0', 'discrete_4', 'categorical_1_Yes', 'categorical_4_Level_C']
    low_importance = [col for col in all_features if col not in high_importance + medium_importance]
    
    # assign weights with random signs
    def random_weight(low, high):
        weight = np.random.uniform(low, high)
        sign = np.random.choice([-1, 1])
        return np.round(sign * weight, 2)

    feature_weights = {}
    for f in high_importance:
        feature_weights[f] = random_weight(2.0, 3.0)
    for f in medium_importance:
        feature_weights[f] = random_weight(1.0, 2.0)
    for f in low_importance:
        feature_weights[f] = random_weight(0.0, 1.0)

    return feature_weights, high_importance, medium_importance, low_importance

def generate_target(df_processed, feature_weights):
    """
    Generate binary target variable using a logistic function.
    """
    all_features = df_processed.columns.to_list()
    X = df_processed[all_features].values
    B = np.array([feature_weights[f] for f in all_features])
    
    # mean predicted probability for a given intercept
    def calculate_mean_prob(intercept):
        logit_p = X @ B + intercept
        p = scipy.special.expit(logit_p)
        return p.mean()

    # binary search to balance classes
    low, high = -10.0, 10.0
    for _ in range(25):
        mid = (low + high)/2
        if calculate_mean_prob(mid) > 0.5:
            high = mid
        else:
            low = mid 
    intercept = (low + high) / 2

    # generate target
    logit_p = X @ B + intercept
    p = scipy.special.expit(logit_p)
    y = np.random.binomial(1, p)
    
    # add target to dataframe
    synthetic_data = df_processed.copy()
    synthetic_data['target'] = y
    
    return synthetic_data

def check_data_quality(synthetic_data):
    """
    Check data quality: types, target distribution balance, multicollinearity.
    """
    # check data types
    data_types = synthetic_data.dtypes
    
    # check target distribution
    target_balance = synthetic_data['target'].mean()
    
    # check multicollinearity
    X_vif = synthetic_data.drop(columns=['target'])
    vif_data = pd.DataFrame()
    vif_data['feature'] = X_vif.columns
    vif_data['VIF'] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
    max_vif = vif_data['VIF'].max()
    
    quality_report = {
        'target_balance': target_balance,
        'max_vif': max_vif,
        'num_features': len(synthetic_data.columns) - 1
    }
    
    return quality_report
    
def split_and_save_data(synthetic_data, iteration, output_dir="data/complete", random_state=57):
    """
    Split data into train and test sets and save to files.
    """
    # stratified split data
    train_data, test_data = train_test_split(
        synthetic_data, 
        test_size=0.3, 
        random_state=random_state, 
        stratify=synthetic_data['target']
    )
    
    # create iteration directory path
    iter_dir = f"{output_dir}/iteration_{iteration}"
    os.makedirs(iter_dir, exist_ok=True)
    
    # save files
    train_data.to_csv(f"{iter_dir}/train_complete.csv", index=False)
    test_data.to_csv(f"{iter_dir}/test_complete.csv", index=False)
    
    print(f"Train shape: {train_data.shape}, Test shape: {test_data.shape}")
    print(f"Train target distribution: {train_data['target'].value_counts(normalize=True)}")
    print(f"Test target distribution: {test_data['target'].value_counts(normalize=True)}")
    
    return train_data, test_data

def generate_synthetic_dataset(iteration, random_seed, n_samples=1000):
    """
    Generate a complete synthetic dataset.
    """
    print(f"Generating synthetic dataset for iteration {iteration} with seed {random_seed}")

    # set random seed for replicability
    np.random.seed(random_seed)
    
    # 1. generate features
    cont_features, disc_features, cat_features = generate_features(n_samples)
    # 2. scale and encode features
    df_processed, cont_cols, disc_cols, cat_cols = standardise_features(cont_features, disc_features, cat_features)
    # 3. assign importance weights
    feature_weights, high, medium, low = assign_feature_importance(df_processed)
    # 4. create the target
    synthetic_data = generate_target(df_processed, feature_weights)
    # 5. check data quality
    quality_report = check_data_quality(synthetic_data)
    print(f"Data quality report: {quality_report}")
    # 6. split and save
    train_data, test_data = split_and_save_data(
        synthetic_data, 
        iteration=iteration, 
        output_dir="data/complete", 
        random_state=random_seed
    )
    
    return train_data, test_data, synthetic_data

if __name__ == "__main__":
    for seed in range(1, 101):
    
        # generate dataset
        train_data, test_data, synthetic_data = generate_synthetic_dataset(
            iteration=seed, 
            random_seed=seed,
            n_samples=1000
        )
        print(f"\n")

    print(f"Successfully generated synthetic data for 100 iterations.")

In [ ]:
# rho = 0.2
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import train_test_split
import scipy.special

def generate_features(n_samples=1000):
    """
    Generate mixed data types: 15 continuous, 5 discrete, 5 categorical
    """
    # generate continuous features
    rho = 0.2  
    mus = np.random.uniform(10, 50, 15)
    sigmas = np.random.uniform(2, 10, 15)

    # covariance matrix
    cov = np.zeros((15, 15))
    np.fill_diagonal(cov, sigmas**2)
    for i in range(15):
        for j in range(i+1, 15):
            cov[i, j] = cov[j, i] = rho * sigmas[i] * sigmas[j]
            
    continuous_data = np.random.multivariate_normal(mus, cov, size=n_samples)
    continuous_features = {}
    for i in range(15):
        continuous_features[f'continuous_{i}'] = continuous_data[:, i]

    # generate discrete features
    discrete_features = {}
    lambdas = [2, 3, 4, 5, 6] 
    for i in range(5):
        discrete_features[f'discrete_{i}'] = np.random.poisson(lambdas[i], n_samples)

    # generate categorical features
    categorical_features = {}
    for i in range(5):
        if i < 4:
            # 4 binary features 
            categories = ['No', 'Yes']
            prob = np.random.uniform(0.3, 0.7)
            probs = [1-prob, prob]
            categorical_features[f'categorical_{i}'] = np.random.choice(categories, size=n_samples, p=probs)
        else: 
            # 1 multinomial feature 
            categories = ['Level_A', 'Level_B', 'Level_C']
            probs = [0.4, 0.35, 0.25]  
            categorical_features[f'categorical_{i}'] = np.random.choice(categories, size=n_samples, p=probs)
    
    return continuous_features, discrete_features, categorical_features

def standardise_features(cont_features, disc_features, cat_features):
    """
    Standardise numerical features and one-hot encode categorical features.
    """
    # create initial dataframe
    df = pd.DataFrame(cont_features)
    for col, values in disc_features.items():
        df[col] = values
    for col, values in cat_features.items():
        df[col] = values
    
    # get column lists
    cont_cols = list(cont_features.keys())
    disc_cols = list(disc_features.keys())
    cat_cols = list(cat_features.keys())
    
    # standardise continuous and discrete features
    scaler = StandardScaler()
    df_scaled_cont_disc = pd.DataFrame(
        scaler.fit_transform(df[cont_cols + disc_cols]),
        columns=cont_cols + disc_cols
    )
    
    # one-hot encode categorical features
    encoder = OneHotEncoder(sparse_output=False, drop='first')  
    df_encoded_cat = pd.DataFrame(
        encoder.fit_transform(df[cat_cols]),
        columns=encoder.get_feature_names_out(cat_cols)
    )
    df_encoded_cat = df_encoded_cat.astype(int)
    
    # combine all features
    df_processed = pd.concat([df_scaled_cont_disc, df_encoded_cat], axis=1)
    
    return df_processed, cont_cols, disc_cols, cat_cols

def assign_feature_importance(df_processed):
    """
    Assign importance weights to features.
    """
    all_features = df_processed.columns.to_list()
    
    # define importance categories
    high_importance = ['continuous_0', 'continuous_3', 'continuous_12', 'discrete_1', 'discrete_3', 'categorical_0_Yes', 'categorical_2_Yes']
    medium_importance = ['continuous_1', 'continuous_5', 'continuous_9', 'continuous_11', 'continuous_13', 'discrete_0', 'discrete_4', 'categorical_1_Yes', 'categorical_4_Level_C']
    low_importance = [col for col in all_features if col not in high_importance + medium_importance]
    
    # assign weights with random signs
    def random_weight(low, high):
        weight = np.random.uniform(low, high)
        sign = np.random.choice([-1, 1])
        return np.round(sign * weight, 2)

    feature_weights = {}
    for f in high_importance:
        feature_weights[f] = random_weight(2.0, 3.0)
    for f in medium_importance:
        feature_weights[f] = random_weight(1.0, 2.0)
    for f in low_importance:
        feature_weights[f] = random_weight(0.0, 1.0)

    return feature_weights, high_importance, medium_importance, low_importance

def generate_target(df_processed, feature_weights):
    """
    Generate binary target variable using a logistic function.
    """
    all_features = df_processed.columns.to_list()
    X = df_processed[all_features].values
    B = np.array([feature_weights[f] for f in all_features])
    
    # mean predicted probability for a given intercept
    def calculate_mean_prob(intercept):
        logit_p = X @ B + intercept
        p = scipy.special.expit(logit_p)
        return p.mean()

    # binary search to balance classes
    low, high = -10.0, 10.0
    for _ in range(25):
        mid = (low + high)/2
        if calculate_mean_prob(mid) > 0.5:
            high = mid
        else:
            low = mid 
    intercept = (low + high) / 2

    # generate target
    logit_p = X @ B + intercept
    p = scipy.special.expit(logit_p)
    y = np.random.binomial(1, p)
    
    # add target to dataframe
    synthetic_data = df_processed.copy()
    synthetic_data['target'] = y
    
    return synthetic_data

def check_data_quality(synthetic_data):
    """
    Check data quality: types, target distribution balance, multicollinearity.
    """
    # check data types
    data_types = synthetic_data.dtypes
    
    # check target distribution
    target_balance = synthetic_data['target'].mean()
    
    # check multicollinearity
    X_vif = synthetic_data.drop(columns=['target'])
    vif_data = pd.DataFrame()
    vif_data['feature'] = X_vif.columns
    vif_data['VIF'] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
    max_vif = vif_data['VIF'].max()
    
    quality_report = {
        'target_balance': target_balance,
        'max_vif': max_vif,
        'num_features': len(synthetic_data.columns) - 1
    }
    
    return quality_report
    
def split_and_save_data(synthetic_data, iteration, output_dir="data/complete", random_state=57):
    """
    Split data into train and test sets and save to files.
    """
    # stratified split data
    train_data, test_data = train_test_split(
        synthetic_data, 
        test_size=0.3, 
        random_state=random_state, 
        stratify=synthetic_data['target']
    )
    
    # create iteration directory path
    iter_dir = f"{output_dir}/iteration_{iteration}"
    os.makedirs(iter_dir, exist_ok=True)
    
    # save files
    train_data.to_csv(f"{iter_dir}/train_complete.csv", index=False)
    test_data.to_csv(f"{iter_dir}/test_complete.csv", index=False)
    
    print(f"Train shape: {train_data.shape}, Test shape: {test_data.shape}")
    print(f"Train target distribution: {train_data['target'].value_counts(normalize=True)}")
    print(f"Test target distribution: {test_data['target'].value_counts(normalize=True)}")
    
    return train_data, test_data

def generate_synthetic_dataset(iteration, random_seed, n_samples=1000):
    """
    Generate a complete synthetic dataset.
    """
    print(f"Generating synthetic dataset for iteration {iteration} with seed {random_seed}")

    # set random seed for replicability
    np.random.seed(random_seed)
    
    # 1. generate features
    cont_features, disc_features, cat_features = generate_features(n_samples)
    # 2. scale and encode features
    df_processed, cont_cols, disc_cols, cat_cols = standardise_features(cont_features, disc_features, cat_features)
    # 3. assign importance weights
    feature_weights, high, medium, low = assign_feature_importance(df_processed)
    # 4. create the target
    synthetic_data = generate_target(df_processed, feature_weights)
    # 5. check data quality
    quality_report = check_data_quality(synthetic_data)
    print(f"Data quality report: {quality_report}")
    # 6. split and save
    train_data, test_data = split_and_save_data(
        synthetic_data, 
        iteration=iteration, 
        output_dir="data/complete", 
        random_state=random_seed
    )
    
    return train_data, test_data, synthetic_data

if __name__ == "__main__":
    for seed in range(1, 101):
    
        # generate dataset
        train_data, test_data, synthetic_data = generate_synthetic_dataset(
            iteration=seed, 
            random_seed=seed,
            n_samples=1000
        )
        print(f"\n")

    print(f"Successfully generated synthetic data for 100 iterations.")

In [ ]:
# rho = 0.4
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import train_test_split
import scipy.special

def generate_features(n_samples=1000):
    """
    Generate mixed data types: 15 continuous, 5 discrete, 5 categorical
    """
    # generate continuous features
    rho = 0.4   
    mus = np.random.uniform(10, 50, 15)
    sigmas = np.random.uniform(2, 10, 15)

    # covariance matrix
    cov = np.zeros((15, 15))
    np.fill_diagonal(cov, sigmas**2)
    for i in range(15):
        for j in range(i+1, 15):
            cov[i, j] = cov[j, i] = rho * sigmas[i] * sigmas[j]
            
    continuous_data = np.random.multivariate_normal(mus, cov, size=n_samples)
    continuous_features = {}
    for i in range(15):
        continuous_features[f'continuous_{i}'] = continuous_data[:, i]

    # generate discrete features
    discrete_features = {}
    lambdas = [2, 3, 4, 5, 6] 
    for i in range(5):
        discrete_features[f'discrete_{i}'] = np.random.poisson(lambdas[i], n_samples)

    # generate categorical features
    categorical_features = {}
    for i in range(5):
        if i < 4:
            # 4 binary features 
            categories = ['No', 'Yes']
            prob = np.random.uniform(0.3, 0.7)
            probs = [1-prob, prob]
            categorical_features[f'categorical_{i}'] = np.random.choice(categories, size=n_samples, p=probs)
        else: 
            # 1 multinomial feature 
            categories = ['Level_A', 'Level_B', 'Level_C']
            probs = [0.4, 0.35, 0.25]  
            categorical_features[f'categorical_{i}'] = np.random.choice(categories, size=n_samples, p=probs)
    
    return continuous_features, discrete_features, categorical_features

def standardise_features(cont_features, disc_features, cat_features):
    """
    Standardise numerical features and one-hot encode categorical features.
    """
    # create initial dataframe
    df = pd.DataFrame(cont_features)
    for col, values in disc_features.items():
        df[col] = values
    for col, values in cat_features.items():
        df[col] = values
    
    # get column lists
    cont_cols = list(cont_features.keys())
    disc_cols = list(disc_features.keys())
    cat_cols = list(cat_features.keys())
    
    # standardise continuous and discrete features
    scaler = StandardScaler()
    df_scaled_cont_disc = pd.DataFrame(
        scaler.fit_transform(df[cont_cols + disc_cols]),
        columns=cont_cols + disc_cols
    )
    
    # one-hot encode categorical features
    encoder = OneHotEncoder(sparse_output=False, drop='first')  
    df_encoded_cat = pd.DataFrame(
        encoder.fit_transform(df[cat_cols]),
        columns=encoder.get_feature_names_out(cat_cols)
    )
    df_encoded_cat = df_encoded_cat.astype(int)
    
    # combine all features
    df_processed = pd.concat([df_scaled_cont_disc, df_encoded_cat], axis=1)
    
    return df_processed, cont_cols, disc_cols, cat_cols

def assign_feature_importance(df_processed):
    """
    Assign importance weights to features.
    """
    all_features = df_processed.columns.to_list()
    
    # define importance categories
    high_importance = ['continuous_0', 'continuous_3', 'continuous_12', 'discrete_1', 'discrete_3', 'categorical_0_Yes', 'categorical_2_Yes']
    medium_importance = ['continuous_1', 'continuous_5', 'continuous_9', 'continuous_11', 'continuous_13', 'discrete_0', 'discrete_4', 'categorical_1_Yes', 'categorical_4_Level_C']
    low_importance = [col for col in all_features if col not in high_importance + medium_importance]
    
    # assign weights with random signs
    def random_weight(low, high):
        weight = np.random.uniform(low, high)
        sign = np.random.choice([-1, 1])
        return np.round(sign * weight, 2)

    feature_weights = {}
    for f in high_importance:
        feature_weights[f] = random_weight(2.0, 3.0)
    for f in medium_importance:
        feature_weights[f] = random_weight(1.0, 2.0)
    for f in low_importance:
        feature_weights[f] = random_weight(0.0, 1.0)

    return feature_weights, high_importance, medium_importance, low_importance

def generate_target(df_processed, feature_weights):
    """
    Generate binary target variable using a logistic function.
    """
    all_features = df_processed.columns.to_list()
    X = df_processed[all_features].values
    B = np.array([feature_weights[f] for f in all_features])
    
    # mean predicted probability for a given intercept
    def calculate_mean_prob(intercept):
        logit_p = X @ B + intercept
        p = scipy.special.expit(logit_p)
        return p.mean()

    # binary search to balance classes
    low, high = -10.0, 10.0
    for _ in range(25):
        mid = (low + high)/2
        if calculate_mean_prob(mid) > 0.5:
            high = mid
        else:
            low = mid 
    intercept = (low + high) / 2

    # generate target
    logit_p = X @ B + intercept
    p = scipy.special.expit(logit_p)
    y = np.random.binomial(1, p)
    
    # add target to dataframe
    synthetic_data = df_processed.copy()
    synthetic_data['target'] = y
    
    return synthetic_data

def check_data_quality(synthetic_data):
    """
    Check data quality: types, target distribution balance, multicollinearity.
    """
    # check data types
    data_types = synthetic_data.dtypes
    
    # check target distribution
    target_balance = synthetic_data['target'].mean()
    
    # check multicollinearity
    X_vif = synthetic_data.drop(columns=['target'])
    vif_data = pd.DataFrame()
    vif_data['feature'] = X_vif.columns
    vif_data['VIF'] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
    max_vif = vif_data['VIF'].max()
    
    quality_report = {
        'target_balance': target_balance,
        'max_vif': max_vif,
        'num_features': len(synthetic_data.columns) - 1
    }
    
    return quality_report
    
def split_and_save_data(synthetic_data, iteration, output_dir="data/complete", random_state=57):
    """
    Split data into train and test sets and save to files.
    """
    # stratified split data
    train_data, test_data = train_test_split(
        synthetic_data, 
        test_size=0.3, 
        random_state=random_state, 
        stratify=synthetic_data['target']
    )
    
    # create iteration directory path
    iter_dir = f"{output_dir}/iteration_{iteration}"
    os.makedirs(iter_dir, exist_ok=True)
    
    # save files
    train_data.to_csv(f"{iter_dir}/train_complete.csv", index=False)
    test_data.to_csv(f"{iter_dir}/test_complete.csv", index=False)
    
    print(f"Train shape: {train_data.shape}, Test shape: {test_data.shape}")
    print(f"Train target distribution: {train_data['target'].value_counts(normalize=True)}")
    print(f"Test target distribution: {test_data['target'].value_counts(normalize=True)}")
    
    return train_data, test_data

def generate_synthetic_dataset(iteration, random_seed, n_samples=1000):
    """
    Generate a complete synthetic dataset.
    """
    print(f"Generating synthetic dataset for iteration {iteration} with seed {random_seed}")

    # set random seed for replicability
    np.random.seed(random_seed)
    
    # 1. generate features
    cont_features, disc_features, cat_features = generate_features(n_samples)
    # 2. scale and encode features
    df_processed, cont_cols, disc_cols, cat_cols = standardise_features(cont_features, disc_features, cat_features)
    # 3. assign importance weights
    feature_weights, high, medium, low = assign_feature_importance(df_processed)
    # 4. create the target
    synthetic_data = generate_target(df_processed, feature_weights)
    # 5. check data quality
    quality_report = check_data_quality(synthetic_data)
    print(f"Data quality report: {quality_report}")
    # 6. split and save
    train_data, test_data = split_and_save_data(
        synthetic_data, 
        iteration=iteration, 
        output_dir="data/complete", 
        random_state=random_seed
    )
    
    return train_data, test_data, synthetic_data

if __name__ == "__main__":
    for seed in range(1, 101):
    
        # generate dataset
        train_data, test_data, synthetic_data = generate_synthetic_dataset(
            iteration=seed, 
            random_seed=seed,
            n_samples=1000
        )
        print(f"\n")

    print(f"Successfully generated synthetic data for 100 iterations.")

In [ ]:
# rho = 0.6
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import train_test_split
import scipy.special

def generate_features(n_samples=1000):
    """
    Generate mixed data types: 15 continuous, 5 discrete, 5 categorical
    """
    # generate continuous features
    rho = 0.6
    mus = np.random.uniform(10, 50, 15)
    sigmas = np.random.uniform(2, 10, 15)

    # covariance matrix
    cov = np.zeros((15, 15))
    np.fill_diagonal(cov, sigmas**2)
    for i in range(15):
        for j in range(i+1, 15):
            cov[i, j] = cov[j, i] = rho * sigmas[i] * sigmas[j]
            
    continuous_data = np.random.multivariate_normal(mus, cov, size=n_samples)
    continuous_features = {}
    for i in range(15):
        continuous_features[f'continuous_{i}'] = continuous_data[:, i]

    # generate discrete features
    discrete_features = {}
    lambdas = [2, 3, 4, 5, 6] 
    for i in range(5):
        discrete_features[f'discrete_{i}'] = np.random.poisson(lambdas[i], n_samples)

    # generate categorical features
    categorical_features = {}
    for i in range(5):
        if i < 4:
            # 4 binary features 
            categories = ['No', 'Yes']
            prob = np.random.uniform(0.3, 0.7)
            probs = [1-prob, prob]
            categorical_features[f'categorical_{i}'] = np.random.choice(categories, size=n_samples, p=probs)
        else: 
            # 1 multinomial feature 
            categories = ['Level_A', 'Level_B', 'Level_C']
            probs = [0.4, 0.35, 0.25]  
            categorical_features[f'categorical_{i}'] = np.random.choice(categories, size=n_samples, p=probs)
    
    return continuous_features, discrete_features, categorical_features

def standardise_features(cont_features, disc_features, cat_features):
    """
    Standardise numerical features and one-hot encode categorical features.
    """
    # create initial dataframe
    df = pd.DataFrame(cont_features)
    for col, values in disc_features.items():
        df[col] = values
    for col, values in cat_features.items():
        df[col] = values
    
    # get column lists
    cont_cols = list(cont_features.keys())
    disc_cols = list(disc_features.keys())
    cat_cols = list(cat_features.keys())
    
    # standardise continuous and discrete features
    scaler = StandardScaler()
    df_scaled_cont_disc = pd.DataFrame(
        scaler.fit_transform(df[cont_cols + disc_cols]),
        columns=cont_cols + disc_cols
    )
    
    # one-hot encode categorical features
    encoder = OneHotEncoder(sparse_output=False, drop='first')  
    df_encoded_cat = pd.DataFrame(
        encoder.fit_transform(df[cat_cols]),
        columns=encoder.get_feature_names_out(cat_cols)
    )
    df_encoded_cat = df_encoded_cat.astype(int)
    
    # combine all features
    df_processed = pd.concat([df_scaled_cont_disc, df_encoded_cat], axis=1)
    
    return df_processed, cont_cols, disc_cols, cat_cols

def assign_feature_importance(df_processed):
    """
    Assign importance weights to features.
    """
    all_features = df_processed.columns.to_list()
    
    # define importance categories
    high_importance = ['continuous_0', 'continuous_3', 'continuous_12', 'discrete_1', 'discrete_3', 'categorical_0_Yes', 'categorical_2_Yes']
    medium_importance = ['continuous_1', 'continuous_5', 'continuous_9', 'continuous_11', 'continuous_13', 'discrete_0', 'discrete_4', 'categorical_1_Yes', 'categorical_4_Level_C']
    low_importance = [col for col in all_features if col not in high_importance + medium_importance]
    
    # assign weights with random signs
    def random_weight(low, high):
        weight = np.random.uniform(low, high)
        sign = np.random.choice([-1, 1])
        return np.round(sign * weight, 2)

    feature_weights = {}
    for f in high_importance:
        feature_weights[f] = random_weight(2.0, 3.0)
    for f in medium_importance:
        feature_weights[f] = random_weight(1.0, 2.0)
    for f in low_importance:
        feature_weights[f] = random_weight(0.0, 1.0)

    return feature_weights, high_importance, medium_importance, low_importance

def generate_target(df_processed, feature_weights):
    """
    Generate binary target variable using a logistic function.
    """
    all_features = df_processed.columns.to_list()
    X = df_processed[all_features].values
    B = np.array([feature_weights[f] for f in all_features])
    
    # mean predicted probability for a given intercept
    def calculate_mean_prob(intercept):
        logit_p = X @ B + intercept
        p = scipy.special.expit(logit_p)
        return p.mean()

    # binary search to balance classes
    low, high = -10.0, 10.0
    for _ in range(25):
        mid = (low + high)/2
        if calculate_mean_prob(mid) > 0.5:
            high = mid
        else:
            low = mid 
    intercept = (low + high) / 2

    # generate target
    logit_p = X @ B + intercept
    p = scipy.special.expit(logit_p)
    y = np.random.binomial(1, p)
    
    # add target to dataframe
    synthetic_data = df_processed.copy()
    synthetic_data['target'] = y
    
    return synthetic_data

def check_data_quality(synthetic_data):
    """
    Check data quality: types, target distribution balance, multicollinearity.
    """
    # check data types
    data_types = synthetic_data.dtypes
    
    # check target distribution
    target_balance = synthetic_data['target'].mean()
    
    # check multicollinearity
    X_vif = synthetic_data.drop(columns=['target'])
    vif_data = pd.DataFrame()
    vif_data['feature'] = X_vif.columns
    vif_data['VIF'] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
    max_vif = vif_data['VIF'].max()
    
    quality_report = {
        'target_balance': target_balance,
        'max_vif': max_vif,
        'num_features': len(synthetic_data.columns) - 1
    }
    
    return quality_report
    
def split_and_save_data(synthetic_data, iteration, output_dir="data/complete", random_state=57):
    """
    Split data into train and test sets and save to files.
    """
    # stratified split data
    train_data, test_data = train_test_split(
        synthetic_data, 
        test_size=0.3, 
        random_state=random_state, 
        stratify=synthetic_data['target']
    )
    
    # create iteration directory path
    iter_dir = f"{output_dir}/iteration_{iteration}"
    os.makedirs(iter_dir, exist_ok=True)
    
    # save files
    train_data.to_csv(f"{iter_dir}/train_complete.csv", index=False)
    test_data.to_csv(f"{iter_dir}/test_complete.csv", index=False)
    
    print(f"Train shape: {train_data.shape}, Test shape: {test_data.shape}")
    print(f"Train target distribution: {train_data['target'].value_counts(normalize=True)}")
    print(f"Test target distribution: {test_data['target'].value_counts(normalize=True)}")
    
    return train_data, test_data

def generate_synthetic_dataset(iteration, random_seed, n_samples=1000):
    """
    Generate a complete synthetic dataset.
    """
    print(f"Generating synthetic dataset for iteration {iteration} with seed {random_seed}")

    # set random seed for replicability
    np.random.seed(random_seed)
    
    # 1. generate features
    cont_features, disc_features, cat_features = generate_features(n_samples)
    # 2. scale and encode features
    df_processed, cont_cols, disc_cols, cat_cols = standardise_features(cont_features, disc_features, cat_features)
    # 3. assign importance weights
    feature_weights, high, medium, low = assign_feature_importance(df_processed)
    # 4. create the target
    synthetic_data = generate_target(df_processed, feature_weights)
    # 5. check data quality
    quality_report = check_data_quality(synthetic_data)
    print(f"Data quality report: {quality_report}")
    # 6. split and save
    train_data, test_data = split_and_save_data(
        synthetic_data, 
        iteration=iteration, 
        output_dir="data/complete", 
        random_state=random_seed
    )
    
    return train_data, test_data, synthetic_data

if __name__ == "__main__":
    for seed in range(1, 6):
    
        # generate dataset
        train_data, test_data, synthetic_data = generate_synthetic_dataset(
            iteration=seed, 
            random_seed=seed,
            n_samples=1000
        )
        print(f"\n")

    print(f"Successfully generated synthetic data for 100 iterations.")

In [ ]:
# rho = 0.7
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import train_test_split
import scipy.special

def generate_features(n_samples=1000):
    """
    Generate mixed data types: 15 continuous, 5 discrete, 5 categorical
    """
    # generate continuous features
    rho = 0.7   
    mus = np.random.uniform(10, 50, 15)
    sigmas = np.random.uniform(2, 10, 15)

    # covariance matrix
    cov = np.zeros((15, 15))
    np.fill_diagonal(cov, sigmas**2)
    for i in range(15):
        for j in range(i+1, 15):
            cov[i, j] = cov[j, i] = rho * sigmas[i] * sigmas[j]
            
    continuous_data = np.random.multivariate_normal(mus, cov, size=n_samples)
    continuous_features = {}
    for i in range(15):
        continuous_features[f'continuous_{i}'] = continuous_data[:, i]

    # generate discrete features
    discrete_features = {}
    lambdas = [2, 3, 4, 5, 6] 
    for i in range(5):
        discrete_features[f'discrete_{i}'] = np.random.poisson(lambdas[i], n_samples)

    # generate categorical features
    categorical_features = {}
    for i in range(5):
        if i < 4:
            # 4 binary features 
            categories = ['No', 'Yes']
            prob = np.random.uniform(0.3, 0.7)
            probs = [1-prob, prob]
            categorical_features[f'categorical_{i}'] = np.random.choice(categories, size=n_samples, p=probs)
        else: 
            # 1 multinomial feature 
            categories = ['Level_A', 'Level_B', 'Level_C']
            probs = [0.4, 0.35, 0.25]  
            categorical_features[f'categorical_{i}'] = np.random.choice(categories, size=n_samples, p=probs)
    
    return continuous_features, discrete_features, categorical_features

def standardise_features(cont_features, disc_features, cat_features):
    """
    Standardise numerical features and one-hot encode categorical features.
    """
    # create initial dataframe
    df = pd.DataFrame(cont_features)
    for col, values in disc_features.items():
        df[col] = values
    for col, values in cat_features.items():
        df[col] = values
    
    # get column lists
    cont_cols = list(cont_features.keys())
    disc_cols = list(disc_features.keys())
    cat_cols = list(cat_features.keys())
    
    # standardise continuous and discrete features
    scaler = StandardScaler()
    df_scaled_cont_disc = pd.DataFrame(
        scaler.fit_transform(df[cont_cols + disc_cols]),
        columns=cont_cols + disc_cols
    )
    
    # one-hot encode categorical features
    encoder = OneHotEncoder(sparse_output=False, drop='first')  
    df_encoded_cat = pd.DataFrame(
        encoder.fit_transform(df[cat_cols]),
        columns=encoder.get_feature_names_out(cat_cols)
    )
    df_encoded_cat = df_encoded_cat.astype(int)
    
    # combine all features
    df_processed = pd.concat([df_scaled_cont_disc, df_encoded_cat], axis=1)
    
    return df_processed, cont_cols, disc_cols, cat_cols

def assign_feature_importance(df_processed):
    """
    Assign importance weights to features.
    """
    all_features = df_processed.columns.to_list()
    
    # define importance categories
    high_importance = ['continuous_0', 'continuous_3', 'continuous_12', 'discrete_1', 'discrete_3', 'categorical_0_Yes', 'categorical_2_Yes']
    medium_importance = ['continuous_1', 'continuous_5', 'continuous_9', 'continuous_11', 'continuous_13', 'discrete_0', 'discrete_4', 'categorical_1_Yes', 'categorical_4_Level_C']
    low_importance = [col for col in all_features if col not in high_importance + medium_importance]
    
    # assign weights with random signs
    def random_weight(low, high):
        weight = np.random.uniform(low, high)
        sign = np.random.choice([-1, 1])
        return np.round(sign * weight, 2)

    feature_weights = {}
    for f in high_importance:
        feature_weights[f] = random_weight(2.0, 3.0)
    for f in medium_importance:
        feature_weights[f] = random_weight(1.0, 2.0)
    for f in low_importance:
        feature_weights[f] = random_weight(0.0, 1.0)

    return feature_weights, high_importance, medium_importance, low_importance

def generate_target(df_processed, feature_weights):
    """
    Generate binary target variable using a logistic function.
    """
    all_features = df_processed.columns.to_list()
    X = df_processed[all_features].values
    B = np.array([feature_weights[f] for f in all_features])
    
    # mean predicted probability for a given intercept
    def calculate_mean_prob(intercept):
        logit_p = X @ B + intercept
        p = scipy.special.expit(logit_p)
        return p.mean()

    # binary search to balance classes
    low, high = -10.0, 10.0
    for _ in range(25):
        mid = (low + high)/2
        if calculate_mean_prob(mid) > 0.5:
            high = mid
        else:
            low = mid 
    intercept = (low + high) / 2

    # generate target
    logit_p = X @ B + intercept
    p = scipy.special.expit(logit_p)
    y = np.random.binomial(1, p)
    
    # add target to dataframe
    synthetic_data = df_processed.copy()
    synthetic_data['target'] = y
    
    return synthetic_data

def check_data_quality(synthetic_data):
    """
    Check data quality: types, target distribution balance, multicollinearity.
    """
    # check data types
    data_types = synthetic_data.dtypes
    
    # check target distribution
    target_balance = synthetic_data['target'].mean()
    
    # check multicollinearity
    X_vif = synthetic_data.drop(columns=['target'])
    vif_data = pd.DataFrame()
    vif_data['feature'] = X_vif.columns
    vif_data['VIF'] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
    max_vif = vif_data['VIF'].max()
    
    quality_report = {
        'target_balance': target_balance,
        'max_vif': max_vif,
        'num_features': len(synthetic_data.columns) - 1
    }
    
    return quality_report
    
def split_and_save_data(synthetic_data, iteration, output_dir="data/complete", random_state=57):
    """
    Split data into train and test sets and save to files.
    """
    # stratified split data
    train_data, test_data = train_test_split(
        synthetic_data, 
        test_size=0.3, 
        random_state=random_state, 
        stratify=synthetic_data['target']
    )
    
    # create iteration directory path
    iter_dir = f"{output_dir}/iteration_{iteration}"
    os.makedirs(iter_dir, exist_ok=True)
    
    # save files
    train_data.to_csv(f"{iter_dir}/train_complete.csv", index=False)
    test_data.to_csv(f"{iter_dir}/test_complete.csv", index=False)
    
    print(f"Train shape: {train_data.shape}, Test shape: {test_data.shape}")
    print(f"Train target distribution: {train_data['target'].value_counts(normalize=True)}")
    print(f"Test target distribution: {test_data['target'].value_counts(normalize=True)}")
    
    return train_data, test_data

def generate_synthetic_dataset(iteration, random_seed, n_samples=1000):
    """
    Generate a complete synthetic dataset.
    """
    print(f"Generating synthetic dataset for iteration {iteration} with seed {random_seed}")

    # set random seed for replicability
    np.random.seed(random_seed)
    
    # 1. generate features
    cont_features, disc_features, cat_features = generate_features(n_samples)
    # 2. scale and encode features
    df_processed, cont_cols, disc_cols, cat_cols = standardise_features(cont_features, disc_features, cat_features)
    # 3. assign importance weights
    feature_weights, high, medium, low = assign_feature_importance(df_processed)
    # 4. create the target
    synthetic_data = generate_target(df_processed, feature_weights)
    # 5. check data quality
    quality_report = check_data_quality(synthetic_data)
    print(f"Data quality report: {quality_report}")
    # 6. split and save
    train_data, test_data = split_and_save_data(
        synthetic_data, 
        iteration=iteration, 
        output_dir="data/complete", 
        random_state=random_seed
    )
    
    return train_data, test_data, synthetic_data

if __name__ == "__main__":
    for seed in range(1, 6):
    
        # generate dataset
        train_data, test_data, synthetic_data = generate_synthetic_dataset(
            iteration=seed, 
            random_seed=seed,
            n_samples=1000
        )
        print(f"\n")

    print(f"Successfully generated synthetic data for 100 iterations.")

In [ ]:
# rho = 0.8 
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import train_test_split
import scipy.special

def generate_features(n_samples=1000):
    """
    Generate mixed data types: 15 continuous, 5 discrete, 5 categorical
    """
    # generate continuous features
    rho = 0.8   
    mus = np.random.uniform(10, 50, 15)
    sigmas = np.random.uniform(2, 10, 15)

    # covariance matrix
    cov = np.zeros((15, 15))
    np.fill_diagonal(cov, sigmas**2)
    for i in range(15):
        for j in range(i+1, 15):
            cov[i, j] = cov[j, i] = rho * sigmas[i] * sigmas[j]
            
    continuous_data = np.random.multivariate_normal(mus, cov, size=n_samples)
    continuous_features = {}
    for i in range(15):
        continuous_features[f'continuous_{i}'] = continuous_data[:, i]

    # generate discrete features
    discrete_features = {}
    lambdas = [2, 3, 4, 5, 6] 
    for i in range(5):
        discrete_features[f'discrete_{i}'] = np.random.poisson(lambdas[i], n_samples)

    # generate categorical features
    categorical_features = {}
    for i in range(5):
        if i < 4:
            # 4 binary features 
            categories = ['No', 'Yes']
            prob = np.random.uniform(0.3, 0.7)
            probs = [1-prob, prob]
            categorical_features[f'categorical_{i}'] = np.random.choice(categories, size=n_samples, p=probs)
        else: 
            # 1 multinomial feature 
            categories = ['Level_A', 'Level_B', 'Level_C']
            probs = [0.4, 0.35, 0.25]  
            categorical_features[f'categorical_{i}'] = np.random.choice(categories, size=n_samples, p=probs)
    
    return continuous_features, discrete_features, categorical_features

def standardise_features(cont_features, disc_features, cat_features):
    """
    Standardise numerical features and one-hot encode categorical features.
    """
    # create initial dataframe
    df = pd.DataFrame(cont_features)
    for col, values in disc_features.items():
        df[col] = values
    for col, values in cat_features.items():
        df[col] = values
    
    # get column lists
    cont_cols = list(cont_features.keys())
    disc_cols = list(disc_features.keys())
    cat_cols = list(cat_features.keys())
    
    # standardise continuous and discrete features
    scaler = StandardScaler()
    df_scaled_cont_disc = pd.DataFrame(
        scaler.fit_transform(df[cont_cols + disc_cols]),
        columns=cont_cols + disc_cols
    )
    
    # one-hot encode categorical features
    encoder = OneHotEncoder(sparse_output=False, drop='first')  
    df_encoded_cat = pd.DataFrame(
        encoder.fit_transform(df[cat_cols]),
        columns=encoder.get_feature_names_out(cat_cols)
    )
    df_encoded_cat = df_encoded_cat.astype(int)
    
    # combine all features
    df_processed = pd.concat([df_scaled_cont_disc, df_encoded_cat], axis=1)
    
    return df_processed, cont_cols, disc_cols, cat_cols

def assign_feature_importance(df_processed):
    """
    Assign importance weights to features.
    """
    all_features = df_processed.columns.to_list()
    
    # define importance categories
    high_importance = ['continuous_0', 'continuous_3', 'continuous_12', 'discrete_1', 'discrete_3', 'categorical_0_Yes', 'categorical_2_Yes']
    medium_importance = ['continuous_1', 'continuous_5', 'continuous_9', 'continuous_11', 'continuous_13', 'discrete_0', 'discrete_4', 'categorical_1_Yes', 'categorical_4_Level_C']
    low_importance = [col for col in all_features if col not in high_importance + medium_importance]
    
    # assign weights with random signs
    def random_weight(low, high):
        weight = np.random.uniform(low, high)
        sign = np.random.choice([-1, 1])
        return np.round(sign * weight, 2)

    feature_weights = {}
    for f in high_importance:
        feature_weights[f] = random_weight(2.0, 3.0)
    for f in medium_importance:
        feature_weights[f] = random_weight(1.0, 2.0)
    for f in low_importance:
        feature_weights[f] = random_weight(0.0, 1.0)

    return feature_weights, high_importance, medium_importance, low_importance

def generate_target(df_processed, feature_weights):
    """
    Generate binary target variable using a logistic function.
    """
    all_features = df_processed.columns.to_list()
    X = df_processed[all_features].values
    B = np.array([feature_weights[f] for f in all_features])
    
    # mean predicted probability for a given intercept
    def calculate_mean_prob(intercept):
        logit_p = X @ B + intercept
        p = scipy.special.expit(logit_p)
        return p.mean()

    # binary search to balance classes
    low, high = -10.0, 10.0
    for _ in range(25):
        mid = (low + high)/2
        if calculate_mean_prob(mid) > 0.5:
            high = mid
        else:
            low = mid 
    intercept = (low + high) / 2

    # generate target
    logit_p = X @ B + intercept
    p = scipy.special.expit(logit_p)
    y = np.random.binomial(1, p)
    
    # add target to dataframe
    synthetic_data = df_processed.copy()
    synthetic_data['target'] = y
    
    return synthetic_data

def check_data_quality(synthetic_data):
    """
    Check data quality: types, target distribution balance, multicollinearity.
    """
    # check data types
    data_types = synthetic_data.dtypes
    
    # check target distribution
    target_balance = synthetic_data['target'].mean()
    
    # check multicollinearity
    X_vif = synthetic_data.drop(columns=['target'])
    vif_data = pd.DataFrame()
    vif_data['feature'] = X_vif.columns
    vif_data['VIF'] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
    max_vif = vif_data['VIF'].max()
    
    quality_report = {
        'target_balance': target_balance,
        'max_vif': max_vif,
        'num_features': len(synthetic_data.columns) - 1
    }
    
    return quality_report
    
def split_and_save_data(synthetic_data, iteration, output_dir="data/complete", random_state=57):
    """
    Split data into train and test sets and save to files.
    """
    # stratified split data
    train_data, test_data = train_test_split(
        synthetic_data, 
        test_size=0.3, 
        random_state=random_state, 
        stratify=synthetic_data['target']
    )
    
    # create iteration directory path
    iter_dir = f"{output_dir}/iteration_{iteration}"
    os.makedirs(iter_dir, exist_ok=True)
    
    # save files
    train_data.to_csv(f"{iter_dir}/train_complete.csv", index=False)
    test_data.to_csv(f"{iter_dir}/test_complete.csv", index=False)
    
    print(f"Train shape: {train_data.shape}, Test shape: {test_data.shape}")
    print(f"Train target distribution: {train_data['target'].value_counts(normalize=True)}")
    print(f"Test target distribution: {test_data['target'].value_counts(normalize=True)}")
    
    return train_data, test_data

def generate_synthetic_dataset(iteration, random_seed, n_samples=1000):
    """
    Generate a complete synthetic dataset.
    """
    print(f"Generating synthetic dataset for iteration {iteration} with seed {random_seed}")

    # set random seed for replicability
    np.random.seed(random_seed)
    
    # 1. generate features
    cont_features, disc_features, cat_features = generate_features(n_samples)
    # 2. scale and encode features
    df_processed, cont_cols, disc_cols, cat_cols = standardise_features(cont_features, disc_features, cat_features)
    # 3. assign importance weights
    feature_weights, high, medium, low = assign_feature_importance(df_processed)
    # 4. create the target
    synthetic_data = generate_target(df_processed, feature_weights)
    # 5. check data quality
    quality_report = check_data_quality(synthetic_data)
    print(f"Data quality report: {quality_report}")
    # 6. split and save
    train_data, test_data = split_and_save_data(
        synthetic_data, 
        iteration=iteration, 
        output_dir="data/complete", 
        random_state=random_seed
    )
    
    return train_data, test_data, synthetic_data

if __name__ == "__main__":
    for seed in range(1, 6):
    
        # generate dataset
        train_data, test_data, synthetic_data = generate_synthetic_dataset(
            iteration=seed, 
            random_seed=seed,
            n_samples=1000
        )
        print(f"\n")

    print(f"Successfully generated synthetic data for 100 iterations.")